# K-Means Clustering with PySpark MLlib

This notebook converts a scikit-learn KMeans workflow into **PySpark MLlib** format using the Iris data set.

The main PySpark steps are:

1. Start a Spark session.
2. Load the Iris data.
3. Assemble numeric columns into one `features` vector.
4. Fit a KMeans model.
5. Evaluate clustering quality.
6. Compare clusters with the original species labels.
7. Visualize the clustering results.

## 1. Install and import required packages

Run the installation cell only if PySpark is not already available in your environment, such as Google Colab.

In [ ]:
# Install PySpark if needed. This is useful in Google Colab.
try:
    import pyspark
except ImportError:
    %pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import pandas as pd
import matplotlib.pyplot as plt

## 2. Start a Spark session

A Spark session is the entry point for using PySpark DataFrames and MLlib.

In [ ]:
spark = SparkSession.builder \
    .appName("Iris_KMeans_MLlib") \
    .getOrCreate()

spark

## 3. Load the Iris data

Update `csv_path` if your file is stored in a different location.  
For Google Colab, mount Google Drive before running this cell.

In [ ]:
# Optional for Google Colab users:
# from google.colab import drive
# drive.mount('/content/drive')

csv_path = "/content/drive/MyDrive/Iris.csv"  # Change this path if needed

data = spark.read.csv(csv_path, header=True, inferSchema=True)
data.show(5)

## 4. Inspect and clean the data

The original Iris CSV may contain spaces in column names, such as ` SepalLengthCm`.  
This cell removes leading and trailing spaces from all column names.

In [ ]:
# Remove leading/trailing spaces from column names
for old_name in data.columns:
    data = data.withColumnRenamed(old_name, old_name.strip())

print(data.columns)
data.printSchema()

In [ ]:
# Basic data checks
print("Number of rows:", data.count())
print("Number of columns:", len(data.columns))

data.describe().show()

In [ ]:
# Check missing values in each column
missing_counts = data.select([
    count(when(col(c).isNull(), c)).alias(c) for c in data.columns
])
missing_counts.show()

## 5. Select feature columns

KMeans is an unsupervised learning method, so the species label is not used for model training.  
The label can still be used later to interpret the clusters.

In [ ]:
feature_cols = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]

# Keep only needed columns
iris_df = data.select(feature_cols + ["Species"])
iris_df.show(5)

## 6. Convert columns to MLlib feature vectors

PySpark MLlib requires the input variables to be stored in a single vector column, usually named `features`.

In [ ]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features"
)

assembled_df = assembler.transform(iris_df)
assembled_df.select("raw_features", "Species").show(5, truncate=False)

## 7. Standardize the features

Standardization is usually recommended for distance-based methods such as KMeans.

In [ ]:
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

scaled_df.select("features", "Species").show(5, truncate=False)

## 8. Fit the KMeans model

The Iris data set has three species, so we use `k = 3` clusters.

In [ ]:
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=3,
    seed=123
)

model = kmeans.fit(scaled_df)
predictions = model.transform(scaled_df)

predictions.select("features", "Species", "cluster").show(10, truncate=False)

## 9. Evaluate clustering performance

The silhouette score measures how similar each point is to its own cluster compared with other clusters.  
A larger silhouette score indicates better separated clusters.

In [ ]:
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score = {silhouette:.4f}")

## 10. Show cluster centers

The cluster centers are reported in the standardized feature space because the model was trained using scaled features.

In [ ]:
centers = model.clusterCenters()

for i, center in enumerate(centers):
    print(f"Cluster {i}: {center}")

## 11. Compare clusters with species labels

KMeans does not know the species labels during training.  
This table helps interpret which species are grouped into each cluster.

In [ ]:
predictions.groupBy("cluster", "Species").count().orderBy("cluster", "Species").show()

## 12. Visualize the clusters

For visualization, we convert the Spark DataFrame to a pandas DataFrame and plot two variables: petal length and petal width.

In [ ]:
plot_df = predictions.select(
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
    "Species",
    "cluster"
).toPandas()

plot_df.head()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    plot_df["PetalLengthCm"],
    plot_df["PetalWidthCm"],
    c=plot_df["cluster"]
)
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.title("PySpark MLlib KMeans Clusters on Iris Data")
plt.show()

## 13. Try different values of k

This section computes silhouette scores for several values of `k`.

In [ ]:
results = []

for k in range(2, 8):
    km = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=123
    )
    km_model = km.fit(scaled_df)
    km_predictions = km_model.transform(scaled_df)
    score = evaluator.evaluate(km_predictions)
    results.append((k, score))

results_df = pd.DataFrame(results, columns=["k", "silhouette"])
results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["k"], results_df["silhouette"], marker="o")
plt.xlabel("Number of clusters, k")
plt.ylabel("Silhouette score")
plt.title("Silhouette Scores for Different k Values")
plt.show()

## 14. Stop the Spark session

Run this cell when you are finished with the notebook.

In [ ]:
spark.stop()